# 02, Feature Engineering Pipeline
**CoolingHealthSentinel Pre-Onboarding, Day 3**

Builds the temporal feature set the production GBR model currently lacks: lag
features, rolling statistics, exponential weighted means, rate-of-change terms,
cyclical time encodings, and physically-motivated interaction terms. Saves the
result as `data/features_engineered.csv` and documents every new feature in
`data/feature_registry.md`.

**Note on this version:** built against the reconciled two-population dataset
(`anomaly_class` = structured / background / none). The feature formulas
themselves don't depend on anomaly class, they're generic transformations of
the raw telemetry, but the NaN-drop step interacts with anomaly placement
differently than before, since background rows are scattered randomly with no
edge-avoidance. That's checked explicitly below rather than assumed unchanged.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data/cooling_telemetry_doha_dc1.csv", parse_dates=["timestamp"])
df["anomaly_type"] = df["anomaly_type"].fillna("none")
print(df.shape)
print(df["anomaly_class"].value_counts())

(35040, 17)
anomaly_class
none          33974
background      943
structured      123
Name: count, dtype: int64


## Lag Features (20)
`lag_1, lag_4, lag_8, lag_16, lag_32` (15 min, 1h, 2h, 4h, 8h back) for
`chiller_cop`, `pump_vibration_mms`, `chiller_inlet_temp_c`, and `it_load_mw`, unchanged rationale from the original version: recent history for the four
columns most directly tied to incipient mechanical/thermal failure.

In [2]:
lag_cols = ["chiller_cop", "pump_vibration_mms", "chiller_inlet_temp_c", "it_load_mw"]
lags = [1, 4, 8, 16, 32]
for c in lag_cols:
    for lag in lags:
        df[f"{c}_lag_{lag}"] = df[c].shift(lag)
print(f"Added {len(lag_cols) * len(lags)} lag features")

Added 20 lag features


## Rolling Statistics (18)
1h / 4h / 24h rolling mean and std for `chiller_cop`, `pump_vibration_mms`, and
`cooling_tower_approach_c`, same three columns as before, still chosen because
Day 2 (rebuilt) confirmed they show the largest z-shift for the structured
population specifically.

In [3]:
roll_cols = ["chiller_cop", "pump_vibration_mms", "cooling_tower_approach_c"]
windows = [(4, "1h"), (16, "4h"), (96, "24h")]
for c in roll_cols:
    for w, label in windows:
        df[f"{c}_roll_{label}_mean"] = df[c].rolling(w).mean()
        df[f"{c}_roll_{label}_std"] = df[c].rolling(w).std()
print(f"Added {len(roll_cols) * len(windows) * 2} rolling features")

Added 18 rolling features


## Exponential Weighted Means (6)
EWM spans 8/32/96 for `chiller_cop` and `pue`, the two strongest CHS correlates.

In [4]:
ewm_cols = ["chiller_cop", "pue"]
spans = [8, 32, 96]
for c in ewm_cols:
    for span in spans:
        df[f"{c}_ewm_{span}"] = df[c].ewm(span=span, adjust=False).mean()
print(f"Added {len(ewm_cols) * len(spans)} EWM features")

Added 6 EWM features


## Rate-of-Change Features (4)
1-step and 4-step diffs for `pump_vibration_mms` and `chiller_outlet_temp_c`, slope, not level, as the early-warning signal.

In [5]:
diff_cols = ["pump_vibration_mms", "chiller_outlet_temp_c"]
for c in diff_cols:
    df[f"{c}_diff_1"] = df[c].diff(1)
    df[f"{c}_diff_4"] = df[c].diff(4)
print(f"Added {len(diff_cols) * 2} rate-of-change features")

Added 4 rate-of-change features


## Time Features (8)
Raw calendar features plus cyclical sin/cos encodings of hour and month.

In [6]:
df["hour_of_day"] = df["timestamp"].dt.hour
df["day_of_week"] = df["timestamp"].dt.dayofweek
df["month_num"] = df["timestamp"].dt.month
df["is_weekend"] = df["day_of_week"].isin([5, 6]).astype(int)

df["sin_hour"] = np.sin(2 * np.pi * df["hour_of_day"] / 24)
df["cos_hour"] = np.cos(2 * np.pi * df["hour_of_day"] / 24)
df["sin_month"] = np.sin(2 * np.pi * df["month_num"] / 12)
df["cos_month"] = np.cos(2 * np.pi * df["month_num"] / 12)
print("Added 8 time features")

Added 8 time features


## Interaction Features (3)
Thermal stress proxy, load-to-flow ratio, fouling risk index, same three,
same justification as the original version (see `feature_registry.md`).

In [7]:
df["cop_x_outdoor_temp"] = df["chiller_cop"] * df["outdoor_temp_c"]
df["load_to_flow_ratio"] = df["it_load_mw"] / df["pump_flow_rate_ls"]
df["approach_x_humidity"] = df["cooling_tower_approach_c"] * df["outdoor_humidity_pct"]
print("Added 3 interaction features")

Added 3 interaction features


## NaN Handling, Log Before Dropping

In [8]:
n_before = len(df)
engineered_cols = [c for c in df.columns if any(s in c for s in ["_lag_", "_roll_", "_ewm_", "_diff_"])]

nan_per_col = df[engineered_cols].isna().sum()
rows_with_nan = df[engineered_cols].isna().any(axis=1).sum()

print("Columns with NaNs (lag/rolling/diff only -- EWM and interaction terms never NaN):")
print(nan_per_col[nan_per_col > 0])
print()
print(f"Rows with at least one NaN among engineered columns: {rows_with_nan} of {n_before}")

Columns with NaNs (lag/rolling/diff only -- EWM and interaction terms never NaN):
chiller_cop_lag_1                          1
chiller_cop_lag_4                          4
chiller_cop_lag_8                          8
chiller_cop_lag_16                        16
chiller_cop_lag_32                        32
pump_vibration_mms_lag_1                   1
pump_vibration_mms_lag_4                   4
pump_vibration_mms_lag_8                   8
pump_vibration_mms_lag_16                 16
pump_vibration_mms_lag_32                 32
chiller_inlet_temp_c_lag_1                 1
chiller_inlet_temp_c_lag_4                 4
chiller_inlet_temp_c_lag_8                 8
chiller_inlet_temp_c_lag_16               16
chiller_inlet_temp_c_lag_32               32
it_load_mw_lag_1                           1
it_load_mw_lag_4                           4
it_load_mw_lag_8                           8
it_load_mw_lag_16                         16
it_load_mw_lag_32                         32
chiller_cop_roll_1

In [9]:
dropped_preview = df[df[engineered_cols].isna().any(axis=1)]
dropped_anomalies = dropped_preview[dropped_preview["is_anomaly"] == 1][["timestamp", "anomaly_class", "anomaly_type"]]
print(f"Anomaly rows among the {len(dropped_preview)} dropped rows: {len(dropped_anomalies)}")
dropped_anomalies

Anomaly rows among the 95 dropped rows: 4


,timestamp,anomaly_class,anomaly_type
23,2025-01-01 05:45:00,background,background_sensor_drift
32,2025-01-01 08:00:00,background,background_sensor_drift
45,2025-01-01 11:15:00,background,background_sensor_drift
91,2025-01-01 22:45:00,background,background_sensor_drift


**Same 95-row mechanism as before (the 24h rolling window is still the binding
constraint), but a different consequence for anomaly coverage this time.** The
8 structured events were scheduled with explicit edge-avoidance (the scheduler
excludes the first 5 and last 10 days of the year), so none of them ever
intersected this window, confirmed again here, 0 structured rows lost. The 943
background rows have no such avoidance, since they're sampled uniformly across
the *entire* remaining-normal pool with no positional constraint, so purely by
chance, **4 of them land in the first 95 rows of January 1st** and get dropped
along with the rest of that window. That's 0.4% of the background population, small, but worth stating directly rather than silently losing 4 ground-truth
anomaly labels without comment. The structured population's "no rows lost"
result earlier wasn't luck, it was a deliberate scheduling choice; the
background population's small loss here is the natural consequence of *not*
having made that same choice for it.

In [10]:
df_clean = df.dropna(subset=engineered_cols).reset_index(drop=True)
n_after = len(df_clean)
print(f"Rows before: {n_before}")
print(f"Rows after:  {n_after}")
print(f"Dropped:     {n_before - n_after} ({(n_before - n_after) / n_before:.3%})")
print()
print(f"Anomaly rows retained: {df_clean['is_anomaly'].sum()} of {df['is_anomaly'].sum()}")
print(df_clean.groupby("anomaly_class")["is_anomaly"].count() if False else df_clean["anomaly_class"].value_counts())

Rows before: 35040
Rows after:  34945
Dropped:     95 (0.271%)

Anomaly rows retained: 1062 of 1066
anomaly_class
none          33883
background      939
structured      123
Name: count, dtype: int64


In [11]:
df_clean.to_csv("../data/features_engineered.csv", index=False)
print("Saved data/features_engineered.csv")
print("Final shape:", df_clean.shape)
print("Total columns:", len(df_clean.columns), "| Engineered columns:", len(engineered_cols) + 8 + 3)

Saved data/features_engineered.csv
Final shape: (34945, 76)
Total columns: 76 | Engineered columns: 59


## Summary

59 new engineered features were added to the original telemetry columns: 20 lag,
18 rolling, 6 EWM, 4 rate-of-change, 8 time, and 3 interaction features, unchanged in formula and count from the original (single-population) version,
since these are generic transformations that don't depend on anomaly labeling.

What *did* change: 95 rows are still dropped for the same structural reason (the
24h rolling window needs 95 prior rows), but this time **4 of the 943 background
anomaly rows are lost to that drop** (0 structured rows lost, same as before).
1,062 of 1,066 total anomaly-flagged rows survive into the modeling dataset. This
is documented rather than smoothed over, since Day 5's IsolationForest
evaluation needs the real surviving ground-truth count, not the pre-drop count.

Every feature is documented with its formula and business justification in
`data/feature_registry.md`. This file (`data/features_engineered.csv`) is the
input for the Day 4 GBR "engineered" model and the Day 5 IsolationForest.